# Bad Characters: <br/>Imperceptible NLP Attacks

See related paper for details.

## Setup
Install dependencies and load pre-treained models.

In [ ]:
%%writefile install-apex.sh

export CUDA_HOME=/usr/local/cuda-10.1
git clone https://github.com/NVIDIA/apex
pip install -v --no-cache-dir --global-option="--cpp_ext" --global-option="--cuda_ext" \
  --global-option="--deprecated_fused_adam" --global-option="--xentropy" \
  --global-option="--fast_multihead_attn" ./apex

In [ ]:
# Inference is faster with APEX installed
# This installation is optional and takes a while
!sh install-apex.sh

In [ ]:
# Install dependencies
!pip install pyarrow
!pip install fairseq
!pip install sacremoses
!pip install fastBPE
!pip install subword_nmt
!pip install textdistance[extras]
!pip install scipy
!pip install requests

## Unknown Characters

Unusual characters, such as zero-width spaces and control sequences, are simply encoded as the `<unk>` character by the FairSeq implementation. This likely generalizes to many other NLP models.

In [1]:
# Define function for decoding from the source dictionary
def src_decode(sentence):
  res = []
  for idx in sentence:
    res.append(en2fr.src_dict.symbols[idx])
  return ' '.join(res)

## Invisible Characters

Certian Unicode chacacters are almost never visually rendered by design. Conveniently, they can be embedded within strings and copied + pasted on most systems. Most NLP models are not trained against these characters, making them  not present in source language dictionaries. Thus, they typically result in an `<unk>` embedded vector.

In [2]:
# Zero width space
ZWSP = chr(0x200B)
# Zero width joiner
ZWJ = chr(0x200D)
# Zero width non-joiner
ZWNJ = chr(0x200C)

print(f"{ZWSP}{ZWJ}")

​‍


## Homoglyphs

The Unicode specification defines several homoglyph documents. The following section retrieves these documents and creates mapping between homoglyph characters.

In [3]:
import requests

confusables = dict()
intentionals = dict()

# Retrieve Unicode Confusable homoglyph characters
conf_resp = requests.get("https://www.unicode.org/Public/security/latest/confusables.txt", stream=True)
for line in conf_resp.iter_lines():
  if len(line):
    line = line.decode('utf-8-sig')
    if line[0] != '#':
      line = line.replace("#*", "#")
      _, line = line.split("#", maxsplit=1)
      if line[3] not in confusables:
        confusables[line[3]] = []
      confusables[line[3]].append(line[7])

# Retrieve Unicode Intentional homoglyph characters
int_resp = requests.get("https://www.unicode.org/Public/security/latest/intentional.txt", stream=True)
for line in int_resp.iter_lines():
  if len(line):
    line = line.decode('utf-8-sig')
    if line[0] != '#':
      line = line.replace("#*", "#")
      _, line = line.split("#", maxsplit=1)
      if line[3] not in intentionals:
        intentionals[line[3]] = []
      intentionals[line[3]].append(line[7])

## Reorderings

Unicode Bidirectional (Bidi) Algorithm override characters can be used to render encodeded characters in any order. The following section defines a function which can generate 2^|n| reordered encodings of a given string of length n which are all rendered the same way in any system built on Google's Chromium.

This [site](https://www.soscisurvey.de/tools/view-chars.php) can be used to visualize the underlying encoding of the text.

In [4]:
# Unicode Bidi override characters
PDF = chr(0x202C)
LRE = chr(0x202A)
RLE = chr(0x202B)
LRO = chr(0x202D)
RLO = chr(0x202E)

PDI = chr(0x2069)
LRI = chr(0x2066)
RLI = chr(0x2067)

class Swap():
    """Represents swapped elements in a string of text."""
    def __init__(self, one, two):
        self.one = one
        self.two = two
    
    def __repr__(self):
        return f"Swap({self.one}, {self.two})"

    def __eq__(self, other):
        return self.one == other.one and self.two == other.two

    def __hash__(self):
        return hash((self.one, self.two))

def some(*els):
    """Returns the arguments as a tuple with Nones removed."""
    return tuple(filter(None, tuple(els)))

def swaps(chars: str) -> set:
    """Generates all possible swaps for a string."""
    def pairs(chars, pre=(), suf=()):
        orders = set()
        for i in range(len(chars)-1):
            prefix = pre + tuple(chars[:i])
            suffix = suf + tuple(chars[i+2:])
            swap = Swap(chars[i+1], chars[i])
            pair = some(prefix, swap, suffix)
            orders.add(pair)
            orders.update(pairs(suffix, pre=some(prefix, swap)))
            orders.update(pairs(some(prefix, swap), suf=suffix))
        return orders
    return pairs(chars) | {tuple(chars)}

def unswap(el: tuple) -> str:
    """Reverts a tuple of swaps to the original string."""
    if isinstance(el, str):
        return el
    elif isinstance(el, Swap):
        return unswap((el.two, el.one))
    else:
        res = ""
        for e in el:
            res += unswap(e)
        return res

def uniswap(els):
    res = ""
    for el in els:
        if isinstance(el, Swap):
            res += uniswap([LRO, LRI, RLO, LRI, el.one, PDI, LRI, el.two, PDI, PDF, PDI, PDF])
        elif isinstance(el, str):
            res += el
        else:
            for subel in el:
                res += uniswap([subel])
    return res

def strings_to_file(file, string):
  with open(file, 'w') as f:
      for swap in swaps(string):
          uni = uniswap(swap)
          print(uni, file=f)

def print_strings(string):
  for swap in swaps(string):
    uni = uniswap(swap)
    print(uni)

## Deletions

Unicode control characters used for deleting text can be encoded into strings. Upon rendering, these control characters are actioned and the appropriate surrounding text is not rendered. Yet, NLP models generally still "see" the surrounding text.

In [5]:
# Backspace character
BKSP = chr(0x8)
# Delete character
DEL = chr(0x7F)
# Carriage return character
CR = chr(0xD)

print(f"{CR}{BKSP}{DEL}")




## Untargeted Integrity Attacks

The performance of various NLP models can be degraded through the use of invisible character, homoglyph, reordering, and deletion attacks. The most effective attacks can be found, independent of the underlying model, using a genetic algorithm.

### Attack Setup

Each attack will be defined as an object and set of contstraints over which a genetic algorithm (differential evolution) will optimize. For these attacks, the visual representation of the input is fixed and the aim of the attack is to determine the imperceptible perturbation for which the supplied model's output will be maximally distant from the output of the unperturbed input.

Each attack will be derived from the following Objective abstract class.

In [6]:
from abc import ABC
from typing import List, Tuple, Callable, Dict
from fairseq.hub_utils import GeneratorHubInterface
from scipy.optimize import NonlinearConstraint, differential_evolution
from textdistance import levenshtein
import numpy as np
from concurrent.futures import ThreadPoolExecutor

class Objective(ABC):
    """Abstract class representing objectives for BERT binary classification."""

    def __init__(self, model: GeneratorHubInterface, tokenizer, input_text: str, max_perturbs: int, distance: Callable[[float, float], float], device: str = "cuda:0"):
        if not model:
            raise ValueError("Must supply model.")
        if not tokenizer:
            raise ValueError("Must supply tokenizer.")
        if not input_text:
            raise ValueError("Must supply input text.")

        self.device = torch.device(device)  # 指定设备
        self.model = model.to(self.device)  # 将模型加载到指定设备
        self.tokenizer = tokenizer  # 分词器
        self.input_text = input_text
        self.max_perturbs = max_perturbs
        self.distance = distance  # 距离函数，例如：abs, L2距离等

        # 获取输入文本的分类输出（概率或 logits）
        self.output = self.predict(input_text)

    def predict(self, text: str) -> float:
        """对输入文本使用BERT模型进行预测，返回概率最高的类别的概率。"""
        with torch.no_grad():
            inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(self.device)  # 输入分配到设备
            logits = self.model(**inputs).logits  # 假设输出为logits
        
            # 将 logits 从 GPU 移到 CPU，以避免占用 CUDA 内存
            logits = logits.cpu()

            # 将输入和输出张量从设备中移除
            del inputs  # 删除输入的张量以释放内存

            # 计算概率
            probs = torch.softmax(logits, dim=-1).squeeze().tolist()
            
            # 获取最大概率的类别
            max_prob = max(probs)  # 获取最大概率
            
            # 删除不再需要的变量，防止占用额外的内存
            del logits
            torch.cuda.empty_cache()  # 如果必须使用的话，清理缓存，但不要频繁调用

            return max_prob  # 返回概率最高的类别的概率



    def objective(self) -> Callable[[List[float]], float]:
        def _objective(perturbations: List[float]) -> float:
            candidate = self.candidate(perturbations)
            candidate_output = self.predict(candidate)
            # 最大化目标类别的概率，取负值作为优化目标
            return -self.distance(self.output, candidate_output)
        return _objective

    def differential_evolution(self, print_result=True, verbose=True, maxiter=60, popsize=32, polish=False, workers=16) -> str:
        executor = ThreadPoolExecutor(max_workers=workers)
        result = differential_evolution(self.objective(), self.bounds(),
                                        disp=verbose, maxiter=maxiter,
                                        popsize=popsize, polish=polish, workers=executor.map)
        candidate = self.candidate(result.x)
        if print_result:
            print(f"Result: {candidate}")
            print(f"Result Distance: {result.fun}")
            print(f"Perturbation Encoding: {result.x}")
            print(f"Input Probability: {self.output}")
            print(f"Result Probability: {self.predict(candidate)}")
        return candidate

    def bounds(self) -> List[Tuple[float, float]]:
        raise NotImplementedError()

    def candidate(self, perturbations: List[float]) -> str:
        raise NotImplementedError()


def natural(x: float) -> int:
    """Rounds float to the nearest natural number (positive int)"""
    return max(0, round(float(x)))

2024-12-01 07:59:52.998164: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-01 07:59:53.015255: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1733039993.035550 3219895 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1733039993.041751 3219895 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-01 07:59:53.063392: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### Machine Translation Attacks

These attacks are against the fariseq EN-FR translation model.

#### Invisible Character Attack

The following attack injects the invisible character supplied to create the maximally effective imperceptible perturbation against the supplied visual input and distance metric.

In [7]:
class InvisibleCharacterObjective(Objective):
    """Class representing an Objective which injects invisible characters."""

    def __init__(self, model: GeneratorHubInterface, tokenizer, input_text: str, max_perturbs: int = 25,
                 invisible_chrs: List[str] = [ZWJ, ZWSP, ZWNJ],
                 distance: Callable[[float, float], float] = lambda x, y: abs(x - y), device: str = "cuda:0"):
        super().__init__(model, tokenizer, input_text, max_perturbs, distance, device=device)
        self.invisible_chrs: List[str] = invisible_chrs

    def bounds(self) -> List[Tuple[float, float]]:
        """
        定义差分进化的边界：
        每个扰动由两个浮点数控制：
        - 第一个浮点数选择插入的不可见字符。
        - 第二个浮点数决定插入的位置。
        """
        return [
            (0, len(self.invisible_chrs) - 1),  # 对应不可见字符的索引
            (-1, len(self.input_text) - 1)     # 对应插入位置的索引（-1 表示不插入）
        ] * self.max_perturbs

    def candidate(self, perturbations: List[float]) -> str:
        """
        根据差分进化生成的扰动，构造新的输入文本。
        """
        candidate = list(self.input_text)  # 转为字符列表
        for i in range(0, len(perturbations), 2):
            # 获取插入的位置和字符索引
            inp_index = natural(perturbations[i + 1])
            if 0 <= inp_index < len(candidate):  # 检查边界
                inv_char = self.invisible_chrs[natural(perturbations[i])]
                candidate.insert(inp_index, inv_char)  # 插入不可见字符
        return ''.join(candidate)

  # def objective(self) -> Callable[[List[float]], float]:
  #     """
  #     定义优化目标函数：
  #     - 使用注入不可见字符的文本计算目标类别概率的变化。
  #     """
  #     def _objective(perturbations: List[float]) -> float:
  #         candidate = self.candidate(perturbations)  # 生成新的输入
  #         candidate_output = self.predict(candidate)  # 计算分类概率
  #         # 最大化目标类别的概率变化，负号表示差分进化最小化
  #         return -self.distance(self.output, candidate_output)
  #     return _objective

#### Homoglyph Attack

This attack replaces characters with homoglyphs to create the maximally effective imperceptible perturbation against the supplied visual input and distance metric.

In [8]:
class HomoglyphObjective(Objective):
  def __init__(self, model: GeneratorHubInterface, tokenizer, input: str, max_perturbs=None, distance: Callable[[str,str],int] = lambda x, y: abs(x - y), homoglyphs: Dict[str,List[str]] = intentionals, device: str = "cuda:0", **kwargs):
    super().__init__(model, tokenizer, input, max_perturbs, distance, device=device)
    if not self.max_perturbs:
      self.max_perturbs = len(self.input_text)
    self.homoglyphs = homoglyphs
    self.glyph_map = []
    for i, char in enumerate(self.input_text):
      if char in self.homoglyphs:
        charmap = self.homoglyphs[char]
        charmap = list(zip([i] * len(charmap), charmap))
        self.glyph_map.extend(charmap)

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1, len(self.glyph_map)-1)] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    candidate = [char for char in self.input_text]  
    for perturb in map(natural, perturbations):
      if perturb >= 0:
        i, char = self.glyph_map[perturb]
        candidate[i] = char
    return ''.join(candidate)

#### Reordering Attack

This attack reorders characters using Unicode Bidi overrides to create the maximally effective imperceptible perturbation against the supplied visual input and distance metric.

The reordering patterns used in this attack were designed to be effective against the Bidi implementation used in the Chromium text rendering engine.

In [9]:
class ReorderObjective(Objective):

  def __init__(self, model: GeneratorHubInterface, tokenizer ,input: str, max_perturbs: int = 50, distance: Callable[[float, float], float] = lambda x, y: abs(x - y), device: str = "cuda:0", **kwargs):
    super().__init__(model, tokenizer, input, max_perturbs, distance, device=device)

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1,len(self.input_text)-1)] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    def swaps(els) -> str:
      res = ""
      for el in els:
          if isinstance(el, Swap):
              res += swaps([LRO, LRI, RLO, LRI, el.one, PDI, LRI, el.two, PDI, PDF, PDI, PDF])
          elif isinstance(el, str):
              res += el
          else:
              for subel in el:
                  res += swaps([subel])
      return res

    _candidate = [char for char in self.input_text]
    for perturb in map(natural, perturbations):
      if perturb >= 0 and len(_candidate) >= 2:
        perturb = min(perturb, len(_candidate) - 2)
        _candidate = _candidate[:perturb] + [Swap(_candidate[perturb+1], _candidate[perturb])] + _candidate[perturb+2:]

    return swaps(_candidate)

#### Deletion Attack

This attack inserts Unicode deletion control characters followed by a supplied character to be deleted to create the maximally effective imperceptible perturbation against the supplied visual input and distance metric.

In [10]:
class DeletionObjective(Objective):
  """Class representing an Objective which injects deletion control characters."""

  def __init__(self, model: GeneratorHubInterface, tokenizer, input: str, max_perturbs: int = 100, distance: Callable[[str,str],int] = lambda x, y: abs(x - y), del_chr: str = BKSP, ins_chr_min: str = '!', ins_chr_max: str = '~',device: str = "cuda:0", **kwargs):
    super().__init__(model, tokenizer, input, max_perturbs, distance, device=device)
    self.del_chr: str = del_chr
    self.ins_chr_min: str = ins_chr_min
    self.ins_chr_max: str = ins_chr_max

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1,len(self.input_text)-1), (ord(self.ins_chr_min),ord(self.ins_chr_max))] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    candidate = [char for char in self.input_text]
    for i in range(0, len(perturbations), 2):
      idx = natural(perturbations[i])
      char = chr(natural(perturbations[i+1]))
      candidate = candidate[:idx] + [char, self.del_chr] + candidate[idx:]
      for j in range(i,len(perturbations), 2):
        perturbations[j] += 2
    return ''.join(candidate)

#### Attack Performance

In this section we plot the results of each of the attacks over pertrubations of fixed sizes.

##### Experiment Setup

In [11]:
# Experiemnt for testing objectives within sliding perturbation bounds
from tqdm.auto import tqdm, trange
import pickle
import pandas as pd
import os

# def experiment(model, objective, source, perturbs, min_perturb, max_perturb, file, maxiter, popsize):
#   for i in trange(min_perturb, max_perturb, desc="Perturbations"):
#     perturbs[str(i)] = dict()
#     for docid, doc in tqdm(source.items(), leave=False, desc="Document"):
#       perturbs[str(i)][docid] = dict()
#       for segid, seg in tqdm(doc.items(), leave=False, desc="Sequence"):
#         perturbs[str(i)][docid][segid] = objective(en2fr, seg, max_perturbs=i).differential_evolution(print_result=False, verbose=False, maxiter=maxiter, popsize=popsize)
#         with open(file, 'wb') as f:
#           pickle.dump(perturbs, f)

def experiment(model,tokenizer, objective, source, perturbs, perturb_list, file, maxiter, popsize, task_name, method_name, output_dir, workers = 16, device = "cuda:0"):
    # 用于保存记录的列表
    records = []
    
    for i in tqdm(perturb_list, desc="Perturbations"):  # 遍历 perturb_list
        perturbs[str(i)] = dict()
        for segid, seg in tqdm(source.items(), leave=False, desc="Sequence"):  # 遍历 source 中的句子
            try:
                perturbs[str(i)][segid] = objective(
                    model,tokenizer, seg, max_perturbs=i, device = device
                ).differential_evolution(print_result=False, verbose=False, maxiter=maxiter, popsize=popsize, workers=workers)
                
                # 记录原始文本和扰动后的文本
                perturbed_text = perturbs[str(i)][segid]
                success = True
            except Exception as e:
                success = False
                perturbed_text = seg
                import traceback
                traceback.print_exc()
                print(e)
                return
            records.append({
                "idx": segid,
                "origin_text": seg,
                "perturb_text": perturbed_text,
                "success": success
            })
            
            # 每次保存perturbs数据
            with open(os.path.join(output_dir, file), 'wb') as f:
                pickle.dump(perturbs, f)
        
        # 根据i值生成文件名
        csv_filename = os.path.join(output_dir, f"{task_name}_Perturb{i}_{method_name}.csv")
        # 将记录保存为CSV文件
        df = pd.DataFrame(records)
        df.to_csv(csv_filename, index=False)
        
        # 清空记录列表以便下一次保存
        records = []

#### Read Dataset

In [12]:
import pandas as pd

# 读取CSV文件
task_name = "toxic"
csv_file = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/data/test.csv"  # 替换为你的实际文件路径
df = pd.read_csv(csv_file)

# 筛选出label为toxic的文本
df_toxic = df[df['label'] == 'toxic']

# 构造 source 和 source_small
source = {str(row['idx']): row['text'] for _, row in df_toxic.iterrows()}
source_small = {k: source[k] for k in list(source.keys())}  # 示例提取部分数据
print(len(source_small))

500


##### Invisible Character Experiment

In [13]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
torch.cuda.set_device(2)
model_path = '/data/rhhuang/models/bert/bert-base-uncased-toxic-batch32-lr5e-05-epochs3/'


tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path, num_labels=2)

In [ ]:
# Invisible Character Experiment
# Only consider a subset (65 examples) of the source
# source_small = { '1016-latimes': source['1016-latimes'], '2327-dailymail.co.uk': source['2327-dailymail.co.uk'] }
perturbs = { '0': source_small }

# experiment(en2fr, InvisibleCharacterObjective, source_small, perturbs, 1, 6, 'invisible_chars.pkl', 5, 16)
output_dir = "/data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation"
method_name = "invisible_chars"
experiment(model, tokenizer, InvisibleCharacterObjective, source_small, perturbs, [1, 2, 4, 8, 16], f'{task_name}_{method_name}.pkl', 5, 16, task_name, method_name, output_dir, workers=2, device = "cuda:2")

Perturbations:   0%|          | 0/5 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

/home/rhhuang/.local/lib/python3.10/site-packages/scipy/optimize/_differentialevolution.py:377: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

##### Homoglyph Experiment

In [14]:
# Homoglyph Experiment
perturbs = { '0': source_small }
output_dir = "/data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation"
method_name = "homoglyphs"
experiment(model,tokenizer, HomoglyphObjective, source_small, perturbs, [1, 2, 4, 8, 16], f'{task_name}_{method_name}.pkl', 5, 16, task_name, method_name, output_dir, workers=2, device = "cuda:2")

Perturbations:   0%|          | 0/5 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

/home/rhhuang/.local/lib/python3.10/site-packages/scipy/optimize/_differentialevolution.py:377: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

##### Reordering Experiment

In [15]:
# Reordering Experiment

# experiment(en2fr, ReorderObjective, source_small, perturbs, 1, 6, 'reorder.pkl', 3, 16)
method_name = "reorder"
experiment(model,tokenizer, ReorderObjective, source_small, perturbs, [1, 2, 4, 8, 16], f'{task_name}_{method_name}.pkl', 5, 16, task_name, method_name, output_dir, workers=2, device = "cuda:2")

Perturbations:   0%|          | 0/5 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

##### Deletion Experiment

In [16]:
# Deletion Experiment

# experiment(en2fr, DeletionObjective, source_small, perturbs, 1, 6, 'deletion.pkl', 3, 16)
method_name = "deletion"
experiment(model,tokenizer, DeletionObjective, source_small, perturbs, [1, 2, 4, 8, 16], f'{task_name}_{method_name}.pkl', 5, 16, task_name, method_name, output_dir, workers=2, device = "cuda:2")

Perturbations:   0%|          | 0/5 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

Sequence:   0%|          | 0/500 [00:00<?, ?it/s]

# Convert Result

In [18]:
import pandas as pd
import os

def convert_csv_with_full_mapping(input_csv_path, output_csv_path, dataset_path, pos_label, label2values):
    # 读取 test.csv 文件，生成映射
    test_df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
    test_df['label_value'] = test_df['label'].map(label2values)
    text_label_map = dict(zip(test_df['text'], test_df['label']))
    text_label_value_map = dict(zip(test_df['text'], test_df['label_value']))
    idx_map = dict(zip(test_df['text'], test_df.index))
    
    # 读取实验生成的CSV文件
    input_df = pd.read_csv(input_csv_path)
    
    # 映射 perturbed_text 的 idx、label、label_value
    input_df['label'] = input_df['origin_text'].map(text_label_map)
    input_df['label_value'] = input_df['origin_text'].map(text_label_value_map)
    input_df['idx'] = input_df['origin_text'].map(idx_map)
    
    # 获取非 pos_label 的数据
    non_pos_data = test_df[test_df['label'] != pos_label].rename(columns={'text': 'perturb_text'})
    non_pos_data['idx'] = non_pos_data.index
    non_pos_data = non_pos_data[['idx', 'perturb_text', 'label_value', 'label']].rename(columns={'perturb_text': 'text'})
    
    # 整合数据
    output_df = pd.concat([
        input_df.rename(columns={'perturb_text': 'text'})[['idx', 'text', 'label_value', 'label']],
        non_pos_data
    ], ignore_index=True)
    
    # 保存为新的CSV文件
    output_df.to_csv(output_csv_path, index=False)
    print(f"转换完成，保存为 {output_csv_path}")


In [24]:
# 输入和输出路径
task_name = 'toxic'
perturbs = [1, 2, 4, 8, 16]
method_names = ["invisible_chars", "homoglyphs", "reorder", "deletion"]
input_csv_path = "/data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation"
output_csv_path = "/data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/"
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/"
pos_label = 'negative'
label2values = {'positive': 1, 'negative': 0}

# 执行转换
for method_name in method_names:
    for perturb in perturbs:
        convert_csv_with_full_mapping(os.path.join(input_csv_path , f"{task_name}_Perturb{perturb}_{method_name}.csv"), os.path.join(output_csv_path ,f"{task_name}_Perturb{perturb}_{method_name}.csv"), dataset_path, pos_label, label2values)


转换完成，保存为 /data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/toxic_Perturb1_invisible_chars.csv
转换完成，保存为 /data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/toxic_Perturb2_invisible_chars.csv
转换完成，保存为 /data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/toxic_Perturb4_invisible_chars.csv
转换完成，保存为 /data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/toxic_Perturb8_invisible_chars.csv
转换完成，保存为 /data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/toxic_Perturb16_invisible_chars.csv
转换完成，保存为 /data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/toxic_Perturb1_homoglyphs.csv
转换完成，保存为 /data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/toxic_Perturb2_homoglyphs.csv
转换完成，保存为 /data/rhhuang/notebooks/AdvTradition/Imperceptible_Perturbation/format_result/toxic_Perturb4_homoglyphs.csv
转换完成，保存为 /data/rhhuang/notebooks/AdvTr

## Load Experiment Result

In [ ]:
import pickle

def load_perturbs(file):
    """
    读取存储的 perturbs 数据。

    参数:
        file (str): 保存 perturbs 数据的文件路径。

    返回:
        dict: 加载的 perturbs 数据。
    """
    with open(file, 'rb') as f:
        perturbs = pickle.load(f)
    return perturbs




In [ ]:
file = "deletion.pkl"
loaded_perturbs = load_perturbs(file)
print(loaded_perturbs)
loaded_perturbs['0']
loaded_perturbs['2']

In [ ]:
source

### MNLI Attacks

#### Attack Setup

In [ ]:
!wget https://cims.nyu.edu/~sbowman/multinli/multinli_1.0.zip
!unzip multinli_1.0.zip
!rm -rf __MACOSX/

In [ ]:
import json
with open('multinli_1.0/multinli_1.0_dev_matched.jsonl', 'r') as f:
  mnli_test = [json.loads(jline) for jline in f.readlines()]

In [ ]:
# Load pre-trained translation model
import torch
mnli = torch.hub.load('pytorch/fairseq',
                       'roberta.large.mnli').eval().cuda()
label_map = {'contradiction': 0, 'neutral': 1, 'entailment': 2}

In [ ]:
class MnliObjective():

  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, max_perturbs: int):
    if not model:
      raise ValueError("Must supply model.")
    if not input:
      raise ValueError("Must supply input.")
    if not hypothesis:
      raise ValueError("Must supply hypothesis.")
    if label == None:
      raise ValueError("Must supply label.")
    self.model: GeneratorHubInterface = model
    self.input: str = input
    self.hypothesis: str = hypothesis
    self.label: int = label
    self.max_perturbs: int = max_perturbs

  def objective(self) -> Callable[[List[float]], float]:
    def _objective(perturbations: List[float]) -> float:
      candidate: str = self.candidate(perturbations)
      tokens = self.model.encode(candidate, self.hypothesis)
      predict = self.model.predict('mnli', tokens)
      if predict.argmax() != self.label:
        return -np.inf
      else:
        return predict.cpu().detach().numpy()[0][self.label]
    return _objective

  def differential_evolution(self, print_result=True, verbose=True, maxiter=3, popsize=32, polish=False) -> str:
    result = differential_evolution(self.objective(), self.bounds(),
                                    disp=verbose, maxiter=maxiter,
                                    popsize=popsize, polish=polish)
    candidate = self.candidate(result.x)
    if (print_result):
      print(f"Result: {candidate}")
      print(f"Correct Label Prediction: {result.fun}")
      print(f"Perturbation Encoding: {result.x}")
    return candidate

#### Invisible Character Attack

In [ ]:
class InvisibleCharacterMnliObjective(MnliObjective, InvisibleCharacterObjective):
  
  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, max_perturbs: int = 10, invisible_chrs: List[str] = [ZWJ,ZWSP,ZWNJ], **kwargs):
    super().__init__(model, input, hypothesis, label, max_perturbs)
    self.invisible_chrs = invisible_chrs

#### Homoglyph Attack

In [ ]:
class HomoglyphMnliObjective(MnliObjective, HomoglyphObjective):
  
  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, max_perturbs: int = 10, homoglyphs: Dict[str,List[str]] = intentionals, **kwargs):
    super().__init__(model, input, hypothesis, label, max_perturbs)
    self.homoglyphs = homoglyphs
    self.glyph_map = []
    for i, char in enumerate(self.input):
      if char in self.homoglyphs:
        charmap = self.homoglyphs[char]
        charmap = list(zip([i] * len(charmap), charmap))
        self.glyph_map.extend(charmap)

#### Reordering Attack

In [ ]:
class ReorderMnliObjective(MnliObjective, ReorderObjective):
  
  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, max_perturbs: int = 10, **kwargs):
    super().__init__(model, input, hypothesis, label, max_perturbs)

#### Deletion Attack

In [ ]:
class DeletionMnliObjective(MnliObjective, DeletionObjective):
  
  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, max_perturbs: int = 10, del_chr: str = BKSP, ins_chr_min: str = '!', ins_chr_max: str = '~', **kwargs):
    super().__init__(model, input, hypothesis, label, max_perturbs)
    self.del_chr: str = del_chr
    self.ins_chr_min: str = ins_chr_min
    self.ins_chr_max: str = ins_chr_max

#### Attack Performance

##### Experiment Setup

In [ ]:
def mnli_experiment(model, objective, data, file, min_budget, max_budget, maxiter, popsize):
  perturbs = { '0': data }
  for budget in trange(min_budget, max_budget):
    perturbs[str(budget)] = dict()
    for test in tqdm(data, leave=False):
      obj = objective(mnli, test['sentence1'], test['sentence2'], label_map[test['gold_label']])
      example = obj.differential_evolution(print_result=False, verbose=False, maxiter=maxiter, popsize=popsize)
      perturbs[str(budget)][test['pairID']] = example
      with open(file, 'wb') as f:
          pickle.dump(perturbs, f)

##### Invisible Character Experiment

In [ ]:
mnli_experiment(mnli, InvisibleCharacterMnliObjective, mnli_test[:20], "mnli_invisible_chars.pkl", 1, 6, 3, 16)

##### Homoglyph Experiment

In [ ]:
mnli_experiment(mnli, HomoglyphMnliObjective, mnli_test[:20], "mnli_homoglyphs.pkl", 1, 6, 3, 16)

##### Reordering Experiment

In [ ]:
mnli_experiment(mnli, ReorderMnliObjective, mnli_test[:20], "mnli_reorder.pkl", 1, 6, 3, 16)

##### Deletion Experiment

In [ ]:
mnli_experiment(mnli, DeletionMnliObjective, mnli_test[:20], "mnli_deletion.pkl", 1, 6, 3, 16)

### IBM Toxic Content Classifier Attack

Untargeted integrity attacks on the open source IBM Toxic Content filter. Details of the model are available [here](https://github.com/IBM/MAX-Toxic-Comment-Classifier).

####Attack Setup

In [ ]:
%%writefile install-apex.sh

export CUDA_HOME=/usr/local/cuda-10.1
git clone https://github.com/NVIDIA/apex
pip install -v --no-cache-dir --global-option="--cpp_ext" --global-option="--cuda_ext" \
  --global-option="--deprecated_fused_adam" --global-option="--xentropy" \
  --global-option="--fast_multihead_attn" ./apex

In [ ]:
!sh install-apex.sh
!rm -rf assets/
!rm -rf toxic/
!mkdir assets
!wget https://max-cdn.cdn.appdomain.cloud/max-toxic-comment-classifier/1.0.0/assets.tar.gz --output-document=assets/assets.tar.gz
!tar -x -C assets/ -f assets/assets.tar.gz -v
!rm assets/assets.tar.gz
!git clone https://github.com/IBM/MAX-Toxic-Comment-Classifier.git
!mv MAX-Toxic-Comment-Classifier toxic
!sed -i 's/==.*//g' toxic/requirements.txt
!pip install -r toxic/requirements.txt
!pip install maxfw
!sed -i 's/from config/from ..config/g' toxic/core/model.py
!sed -i 's/from core\./from ./g' toxic/core/model.py

In [ ]:
from toxic.core.model import ModelWrapper
from logging import getLogger, WARNING
import torch
getLogger().setLevel(WARNING)
toxic = ModelWrapper()
if torch.cuda.is_available():
  toxic.model.cuda()
  toxic.device = torch.device("cuda")

In [ ]:
class ToxicObjective(ABC):
  """ Toxic content filter objective. """

  def __init__(self, model, input: str, max_perturbs: int):
    if not model:
      raise ValueError("Must supply model.")
    if not input:
      raise ValueError("Must supply input.")

    self.model = model
    self.input: str = input
    self.max_perturbs: int = max_perturbs
    self.output = self.model.predict([self.input])[0]

  def objective(self) -> Callable[[List[float]], float]:
    def _objective(perturbations: List[float]) -> float:
      candidate: str = self.candidate(perturbations)
      predicts: str = self.model.predict([candidate])[0]
      return sum(predicts.values())
    return _objective

  def differential_evolution(self, print_result=True, verbose=True, maxiter=60, popsize=32, polish=False) -> str:
    result = differential_evolution(self.objective(), self.bounds(),
                                    disp=verbose, maxiter=maxiter,
                                    popsize=popsize, polish=polish)
    candidate = self.candidate(result.x)
    if (print_result):
      print(f"Result: {candidate}")
      print(f"Result Predictions: {self.model.predict([candidate])[0]}")
      print(f"Input: {self.input}")
      print(f"Input Predictions: {self.output}")
    return candidate

  def bounds(self) -> List[Tuple[float, float]]:
    raise NotImplementedError()

  def candidate(self, perturbations: List[float]) -> str:
    raise NotImplementedError()

####Invisible Character Attack

In [ ]:
class InvisibleToxicObjective(ToxicObjective):
  """Class representing a Toxic Objective which injects invisible characters."""

  def __init__(self, model, input: str, max_perturbs: int = 25, invisible_chrs: List[str] = [ZWJ,ZWSP,ZWNJ], **kwargs):
    super().__init__(model, input, max_perturbs)
    self.invisible_chrs: List[str] = invisible_chrs

  def bounds(self) -> List[Tuple[float, float]]:
    return [(0,len(self.invisible_chrs)-1), (-1, len(self.input)-1)] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    candidate = [char for char in self.input]
    for i in range(0, len(perturbations), 2):
      inp_index = natural(perturbations[i+1])
      if inp_index >= 0:
        inv_char = self.invisible_chrs[natural(perturbations[i])]
        candidate = candidate[:inp_index] + [inv_char] + candidate[inp_index:]
    return ''.join(candidate)

####Homoglyph Attack

In [ ]:
class HomoglyphToxicObjective(ToxicObjective):
  """Class representing a Toxic Objective which injects homoglyphs."""

  def __init__(self, model, input: str, max_perturbs: int = 25, homoglyphs: Dict[str,List[str]] = intentionals, **kwargs):
    super().__init__(model, input, max_perturbs)
    self.homoglyphs = homoglyphs
    self.glyph_map = []
    for i, char in enumerate(self.input):
      if char in self.homoglyphs:
        charmap = self.homoglyphs[char]
        charmap = list(zip([i] * len(charmap), charmap))
        self.glyph_map.extend(charmap)

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1, len(self.glyph_map)-1)] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    candidate = [char for char in self.input]  
    for perturb in map(natural, perturbations):
      if perturb >= 0:
        i, char = self.glyph_map[perturb]
        candidate[i] = char
    return ''.join(candidate)

####Reordering Attack

In [ ]:
class ReorderToxicObjective(ToxicObjective):
  """Class representing a Toxic Objective which injects homoglyphs."""

  def __init__(self, model, input: str, max_perturbs: int = 25, **kwargs):
    super().__init__(model, input, max_perturbs)

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1,len(self.input)-1)] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    def swaps(els) -> str:
      res = ""
      for el in els:
          if isinstance(el, Swap):
              res += swaps([LRO, LRI, RLO, LRI, el.one, PDI, LRI, el.two, PDI, PDF, PDI, PDF])
          elif isinstance(el, str):
              res += el
          else:
              for subel in el:
                  res += swaps([subel])
      return res

    _candidate = [char for char in self.input]
    for perturb in map(natural, perturbations):
      if perturb >= 0 and len(_candidate) >= 2:
        perturb = min(perturb, len(_candidate) - 2)
        _candidate = _candidate[:perturb] + [Swap(_candidate[perturb+1], _candidate[perturb])] + _candidate[perturb+2:]

    return swaps(_candidate)

####Deletion Attack

In [ ]:
class DeletionToxicObjective(ToxicObjective):
  """Class representing a Toxic Objective which injects homoglyphs."""

  def __init__(self, model, input: str, max_perturbs: int = 25, del_chr: str = BKSP, ins_chr_min: str = '!', ins_chr_max: str = '~', **kwargs):
    super().__init__(model, input, max_perturbs)
    self.del_chr = del_chr
    self.ins_chr_min: str = ins_chr_min
    self.ins_chr_max: str = ins_chr_max

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1,len(self.input)-1), (ord(self.ins_chr_min),ord(self.ins_chr_max))] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    candidate = [char for char in self.input]
    for i in range(0, len(perturbations), 2):
      idx = natural(perturbations[i])
      char = chr(natural(perturbations[i+1]))
      candidate = candidate[:idx] + [char, self.del_chr] + candidate[idx:]
      for j in range(i,len(perturbations), 2):
        perturbations[j] += 2
    return ''.join(candidate)

####Attack Performance

#####Experiment Setup

In [ ]:
# Data set details: https://meta.wikimedia.org/wiki/Research:Detox/Data_Release
!wget https://ndownloader.figshare.com/files/7394542 -O toxicity_annotated_comments.tsv
!wget https://ndownloader.figshare.com/files/7394539 -O toxicity_annotations.tsv

In [ ]:
# Load Test Data
import pandas as pd

# load tsv files
comments = pd.read_csv('toxicity_annotated_comments.tsv', sep = '\t', index_col = 0)
annotations = pd.read_csv('toxicity_annotations.tsv',  sep = '\t')
# labels a comment as toxic if the majority of annoatators did so
labels = annotations.groupby('rev_id')['toxicity'].mean() > 0.5
# join labels and comments
comments['toxicity'] = labels
# remove newline and tab tokens
comments['comment'] = comments['comment'].apply(lambda x: x.replace("NEWLINE_TOKEN", " "))
comments['comment'] = comments['comment'].apply(lambda x: x.replace("TAB_TOKEN", " "))
test_comments = comments.query("split=='test'").query("toxicity==True")
examples = test_comments.reset_index().to_dict('records')

In [ ]:
def max_toxic_experiment(objective, model, file, min_budget, max_budget, examples, maxiter, popsize):
  perturbs = { '0': examples }
  with tqdm(total=len(examples)*(max_budget-min_budget+1), desc="Adv. Examples") as pbar:
    for budget in range(min_budget, max_budget+1):
      perturbs[str(budget)] = dict()
      for example in examples:
        obj = objective(model, example['comment'], example['toxicity'], budget)
        result = obj.differential_evolution(verbose=False, print_result=False, maxiter=maxiter, popsize=popsize)
        perturbs[str(budget)][example['rev_id']] = result
        with open(file, 'wb') as f:
          pickle.dump(perturbs, f)
        pbar.update(1)

#####Invisible Character Experiment

In [ ]:
max_toxic_experiment(InvisibleToxicObjective, toxic, 'maxtoxic_invisibles_small.pkl',
                     1, 5, examples[:20], 3, 32)

#####Homoglyph Experiment

In [ ]:
max_toxic_experiment(HomoglyphToxicObjective, toxic, 'maxtoxic_homoglyphs_small.pkl',
                     1, 5, examples[:20], 3, 32)

#####Reordering Experiment

In [ ]:
max_toxic_experiment(ReorderToxicObjective, toxic, 'maxtoxic_reorderings_small.pkl',
                     1, 5, examples[:20], 3, 32)

#####Deletion Experiment

In [ ]:
max_toxic_experiment(DeletionToxicObjective, toxic, 'maxtoxic_deletions_small.pkl',
                     1, 5, examples[:20], 3, 32)

###Perspective API Toxic Content Classifier Attack

Perspective API is a toxic content classifier offered by Google. Details are available [here](https://www.perspectiveapi.com/). You will need to register and get an API key to run these experiments.

####Attack Setup

In [ ]:
import time

class ToxicPerspectiveObjective(ABC):
  """Class representing attacks against Google's toxic language Perspective API."""

  def __init__(self, client, input: str, max_perturbs: int, annotated_toxic: bool):
    if not client:
      raise ValueError("Must supply client.")
    if not input:
      raise ValueError("Must supply input.")

    self.client = client
    self.input: str = input
    self.max_perturbs: int = max_perturbs
    self.annotated_toxic = annotated_toxic
    self.last_request = time.time()
    # Perspective API rate limit in seconds
    self.rate_limit = 0.105
    self.output = self.request(input)

  def request(self, candidate: str):
    delta = time.time() - self.last_request
    if delta < self.rate_limit:
      time.sleep(self.rate_limit - delta)
    self.last_request = time.time()
    analyze_request = {
        'comment': { 'text': candidate },
        'requestedAttributes': {'TOXICITY': {}},
        'languages': 'en',
      }
    response = self.client.comments().analyze(body=analyze_request).execute()
    return response['attributeScores']['TOXICITY']['summaryScore']['value']

  def objective(self) -> Callable[[List[float]], float]:
    def _objective(perturbations: List[float]) -> float:
      candidate: str = self.candidate(perturbations)
      return self.request(candidate)
    return _objective
    
  def differential_evolution(self, print_result=True, verbose=True, maxiter=60, popsize=32, polish=False) -> str:
    start = time.process_time()
    result = differential_evolution(self.objective(), self.bounds(),
                                    disp=verbose, maxiter=maxiter,
                                    popsize=popsize, polish=polish)
    end = time.process_time()
    candidate = self.candidate(result.x)
    adv_output = self.request(candidate)
    if (print_result):
      print(f"Result: {candidate}")
      print(f"Result Predictions: {adv_output}")
      print(f"Input: {self.input}")
      print(f"Input Predictions: {self.output}")
    return candidate

  def bounds(self) -> List[Tuple[float, float]]:
    raise NotImplementedError()

  def candidate(self, perturbations: List[float]) -> str:
    raise NotImplementedError()

In [ ]:
from googleapiclient.discovery_cache.base import Cache
from googleapiclient import discovery
from getpass import getpass
import json


class MemoryCache(Cache):
    _CACHE = {}

    def get(self, url):
        return MemoryCache._CACHE.get(url)

    def set(self, url, content):
        MemoryCache._CACHE[url] = content


perspective = discovery.build(
  "commentanalyzer",
  "v1alpha1",
  developerKey=getpass("Perspective API Key: "),
  discoveryServiceUrl="https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1",
  cache=MemoryCache()
)

####Invisisble Character Attack

In [ ]:
class InvisibleToxicPerspectiveObjective(ToxicPerspectiveObjective):
  """Class representing a Toxic Perspective API Objective which injects invisible characters."""

  def __init__(self, client, input: str, annotated_toxic: bool, max_perturbs: int = 25, invisible_chrs: List[str] = [ZWJ,ZWSP,ZWNJ], **kwargs):
    super().__init__(client, input, max_perturbs, annotated_toxic)
    self.invisible_chrs: List[str] = invisible_chrs

  def bounds(self) -> List[Tuple[float, float]]:
    return [(0,len(self.invisible_chrs)-1), (-1, len(self.input)-1)] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    candidate = [char for char in self.input]
    for i in range(0, len(perturbations), 2):
      inp_index = natural(perturbations[i+1])
      if inp_index >= 0:
        inv_char = self.invisible_chrs[natural(perturbations[i])]
        candidate = candidate[:inp_index] + [inv_char] + candidate[inp_index:]
    return ''.join(candidate)

####Homoglyph Attack

In [ ]:
class HomoglyphToxicPerspectiveObjective(ToxicPerspectiveObjective):
  """Class representing a Toxic Perspective API Objective which injects homoglyphs."""

  def __init__(self, client, input: str, annotated_toxic: bool, max_perturbs: int = 25, homoglyphs: Dict[str,List[str]] = intentionals, **kwargs):
    super().__init__(client, input, max_perturbs, annotated_toxic)
    self.homoglyphs = homoglyphs
    self.glyph_map = []
    for i, char in enumerate(self.input):
      if char in self.homoglyphs:
        charmap = self.homoglyphs[char]
        charmap = list(zip([i] * len(charmap), charmap))
        self.glyph_map.extend(charmap)

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1, len(self.glyph_map)-1)] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    candidate = [char for char in self.input]  
    for perturb in map(natural, perturbations):
      if perturb >= 0:
        i, char = self.glyph_map[perturb]
        candidate[i] = char
    return ''.join(candidate)

####Reordering Attack

In [ ]:
class ReorderToxicPerspectiveObjective(ToxicPerspectiveObjective):
  """Class representing a Toxic Perspective API Objective which injects reorderings."""

  def __init__(self, client, input: str, annotated_toxic: bool, max_perturbs: int = 25, **kwargs):
    super().__init__(client, input, max_perturbs, annotated_toxic)

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1,len(self.input)-1)] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    def swaps(els) -> str:
      res = ""
      for el in els:
          if isinstance(el, Swap):
              res += swaps([LRO, LRI, RLO, LRI, el.one, PDI, LRI, el.two, PDI, PDF, PDI, PDF])
          elif isinstance(el, str):
              res += el
          else:
              for subel in el:
                  res += swaps([subel])
      return res

    _candidate = [char for char in self.input]
    for perturb in map(natural, perturbations):
      if perturb >= 0 and len(_candidate) >= 2:
        perturb = min(perturb, len(_candidate) - 2)
        _candidate = _candidate[:perturb] + [Swap(_candidate[perturb+1], _candidate[perturb])] + _candidate[perturb+2:]

    return swaps(_candidate)

####Deletion Attack

In [ ]:
class DeletionToxicPerspectiveObjective(ToxicPerspectiveObjective):
  """Class representing a Toxic Perspective API Objective which injects deletions."""

  def __init__(self, client, input: str, annotated_toxic: bool, max_perturbs: int = 25, del_chr: str = BKSP, ins_chr_min: str = '!', ins_chr_max: str = '~', **kwargs):
    super().__init__(client, input, max_perturbs, annotated_toxic)
    self.del_chr = del_chr
    self.ins_chr_min: str = ins_chr_min
    self.ins_chr_max: str = ins_chr_max

  def bounds(self) -> List[Tuple[float, float]]:
    return [(-1,len(self.input)-1), (ord(self.ins_chr_min),ord(self.ins_chr_max))] * self.max_perturbs

  def candidate(self, perturbations: List[float]) -> str:
    candidate = [char for char in self.input]
    for i in range(0, len(perturbations), 2):
      idx = natural(perturbations[i])
      char = chr(natural(perturbations[i+1]))
      candidate = candidate[:idx] + [char, self.del_chr] + candidate[idx:]
      for j in range(i,len(perturbations), 2):
        perturbations[j] += 2
    return ''.join(candidate)

####Attack Performance

#####Experiment Setup

In [ ]:
def perspective_experiment(objective, client, file, min_budget, max_budget, examples, maxiter, popsize):
  perturbs = { '0': examples }
  with tqdm(total=len(examples)*(max_budget-min_budget+1), desc="Adv. Examples") as pbar:
    for budget in range(min_budget, max_budget+1):
      perturbs[str(budget)] = dict()
      for example in examples:
        obj = objective(client, example['comment'], budget)
        result = obj.differential_evolution(verbose=False, print_result=False, maxiter=maxiter, popsize=popsize)
        perturbs[str(budget)][example['rev_id']] = result
        with open(file, 'wb') as f:
          pickle.dump(perturbs, f)
        pbar.update(1)

#####Invisible Character Experiment

In [ ]:
perspective_experiment(InvisibleToxicPerspectiveObjective, perspective, 'perspective_invisibles_small_20.pkl',
                       1, 5, examples[:20], 3, 32)

#####Homoglyph Experiment

In [ ]:
perspective_experiment(HomoglyphToxicPerspectiveObjective, perspective, 'perspective_homoglyphs_small_20.pkl',
                       1, 5, examples[:20], 3, 32)

#####Reordering Experiment

In [ ]:
perspective_experiment(ReorderToxicPerspectiveObjective, perspective, 'perspective_reoderings_small_20.pkl',
                       1, 5, examples[:20], 3, 32)

#####Deletion Experiment

In [ ]:
perspective_experiment(DeletionToxicPerspectiveObjective, perspective, 'perspective_deletions_small_20.pkl',
                       1, 5, examples[:20], 3, 32)

## Targeted Integrity Attacks

Targeted imperceptible perturbation attacks craft imperceptible perturbations for a given input that attempt to prodice a fixed output against a given model.

###MNLI Targeted Attack

These attacks target the MNLI textual entailment classification task in a black box model that does have access to the resulting logits for each class during inference.

####Experiment Setup

In [ ]:
from torch.nn.functional import softmax

class MnliTargetedObjective(MnliObjective):

  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label: int, target: int, max_perturbs: int):
    super().__init__(model, input, hypothesis, label, max_perturbs)
    self.target = target

  def objective(self) -> Callable[[List[float]], float]:
      def _objective(perturbations: List[float]) -> float:
        candidate: str = self.candidate(perturbations)
        tokens = self.model.encode(candidate, self.hypothesis)
        predict = self.model.predict('mnli', tokens)
        return -softmax(predict, dim=1).cpu().detach().numpy()[0][self.target]
      return _objective

  def differential_evolution(self, print_result=True, verbose=True, maxiter=3, popsize=32, polish=False) -> str:
    result = differential_evolution(self.objective(), self.bounds(),
                                    disp=verbose, maxiter=maxiter,
                                    popsize=popsize, polish=polish)
    candidate = self.candidate(result.x)
    if (print_result):
      print(f"Result: {candidate}")
      print(f"Correct Label Prediction: {result.fun}")
      print(f"Perturbation Encoding: {result.x}")
    return candidate

####Invisible Character Attack

In [ ]:
class InvisibleCharacterTargetedMnliObjective(MnliTargetedObjective, InvisibleCharacterObjective):
  
  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, target: int, max_perturbs: int = 10, invisible_chrs: List[str] = [ZWJ,ZWSP,ZWNJ], **kwargs):
    super().__init__(model, input, hypothesis, label, target, max_perturbs)
    self.invisible_chrs = invisible_chrs

####Homoglyph Attack

In [ ]:
class HomoglyphTargetedMnliObjective(MnliTargetedObjective, HomoglyphObjective):
  
  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, target: int, max_perturbs: int = 10, homoglyphs: Dict[str,List[str]] = intentionals, **kwargs):
    super().__init__(model, input, hypothesis, label, target, max_perturbs)
    self.homoglyphs = homoglyphs
    self.glyph_map = []
    for i, char in enumerate(self.input):
      if char in self.homoglyphs:
        charmap = self.homoglyphs[char]
        charmap = list(zip([i] * len(charmap), charmap))
        self.glyph_map.extend(charmap)

####Reordering Attack

In [ ]:
class ReorderTargetedMnliObjective(MnliTargetedObjective, ReorderObjective):
  
  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, target: int, max_perturbs: int = 10, **kwargs):
    super().__init__(model, input, hypothesis, label, target, max_perturbs)

####Deletion Attack

In [ ]:
class DeletionTargetedMnliObjective(MnliTargetedObjective, DeletionObjective):
  
  def __init__(self, model: GeneratorHubInterface, input: str, hypothesis: str, label:int, target: int, max_perturbs: int = 10, del_chr: str = BKSP, ins_chr_min: str = '!', ins_chr_max: str = '~', **kwargs):
    super().__init__(model, input, hypothesis, label, target, max_perturbs)
    self.del_chr = del_chr
    self.ins_chr_min: str = ins_chr_min
    self.ins_chr_max: str = ins_chr_max

####Attack Performance

#####Experiment Setup

In [ ]:
from tqdm.auto import tqdm
from time import process_time
import pickle

def mnli_targeted_experiment(objective, model, label_map, inputs, file, min_budget = 10, max_budget = 10, maxiter = 10, popsize = 32):
  results = { '0': inputs }
  with tqdm(total=len(inputs)*(max_budget-min_budget+1)*len(label_map), desc="Adv. Examples") as pbar:
    for budget in range(min_budget, max_budget+1):
      results[str(budget)] = {}
      for input in inputs:
        results[str(budget)][input['pairID']] = []
        for label in range(len(label_map)):
          obj = objective(model, input['sentence1'], input['sentence2'], label_map[input['gold_label']], label, max_perturbs=budget)
          candidate = obj.differential_evolution(print_result=False, verbose=False, maxiter=maxiter, popsize=popsize)
          results[str(budget)][input['pairID']].append(candidate)
          with open(file, 'wb') as f:
            pickle.dump(results, f)
          pbar.update(1)

#####Invisible Character Experiment

In [ ]:
mnli_targeted_experiment(InvisibleCharacterTargetedMnliObjective, mnli, label_map, mnli_test[:20], "mnli_invisibles_targeted.pkl", maxiter=3)

#####Homoglyph Experiment

In [ ]:
mnli_targeted_experiment(HomoglyphTargetedMnliObjective, mnli, label_map, mnli_test[:20], "mnli_homoglyphs_targeted.pkl", maxiter=3)

#####Redordering Experiment

In [ ]:
mnli_targeted_experiment(ReorderTargetedMnliObjective, mnli, label_map, mnli_test[:20], "mnli_reorderings_targeted.pkl", maxiter=3)

#####Deletion Experiment

In [ ]:
mnli_targeted_experiment(DeletionTargetedMnliObjective, mnli, label_map, mnli_test[:20], "mnli_deletions_targeted.pkl", maxiter=3)

###MNLI (No Logits) Targeted Attack

This attack is identical to the MNLI targeted attack except that the adversary only has access to the predicted class and does not have access to the resulting logits for each class during inference.

####Attack Setup

In [ ]:
class MnliTargetedNoLogitsObjective(MnliTargetedObjective):

  def objective(self) -> Callable[[List[float]], float]:
      def _objective(perturbations: List[float]) -> float:
        candidate: str = self.candidate(perturbations)
        tokens = self.model.encode(candidate, self.hypothesis)
        predict = self.model.predict('mnli', tokens)
        if predict.argmax().item() == self.target:
          return -np.inf
        else:
          return np.inf
      return _objective

####Invisible Character Attack

In [ ]:
class InvisibleCharacterTargetedMnliNoLogitsObjective(MnliTargetedNoLogitsObjective, InvisibleCharacterTargetedMnliObjective):
  pass

####Homoglyph Attack

In [ ]:
class HomoglyphTargetedMnliNoLogitsObjective(MnliTargetedNoLogitsObjective, HomoglyphTargetedMnliObjective):
  pass

####Reordering Attack

In [ ]:
class ReorderTargetedMnliNoLogitsObjective(MnliTargetedNoLogitsObjective, ReorderTargetedMnliObjective):
  pass

####Deletion Attack

In [ ]:
class DeletionTargetedMnliNoLogitsObjective(MnliTargetedNoLogitsObjective, DeletionTargetedMnliObjective):
  pass

####Attack Performance

#####Invisible Character Experiment

In [ ]:
mnli_targeted_experiment(InvisibleCharacterTargetedMnliNoLogitsObjective, mnli, label_map, mnli_test[:20], "mnli_invisibles_targeted_nologits.pkl", maxiter=3)

#####Homoglyph Experiment

In [ ]:
mnli_targeted_experiment(HomoglyphTargetedMnliNoLogitsObjective, mnli, label_map, mnli_test[:20], "mnli_homoglyphs_targeted_nologits.pkl", maxiter=3)

#####Reordering Experiment

In [ ]:
mnli_targeted_experiment(ReorderTargetedMnliNoLogitsObjective, mnli, label_map, mnli_test[:20], "mnli_reorderings_targeted_nologits.pkl", maxiter=3)

#####Deletion Exeriment

In [ ]:
mnli_targeted_experiment(DeletionTargetedMnliNoLogitsObjective, mnli, label_map, mnli_test[:20], "mnli_deletions_targeted_nologits.pkl", maxiter=3)

## Availability Attacks

Sponge examples can be crafted from imperceptible perturbations. These examples are optimized to maximize inference compute time. When sent in large batches to ML systems, these examples cam be used to mount an ML denial of service (DoS) availability attack.

### Machine Translation Attacks

####Attack Setup

In [ ]:
from timeit import timeit

class SpongeObjective(Objective):

  def objective(self) -> Callable[[List[float]], float]:
    def _objective(perturbations: List[float]) -> float:
      candidate: str = self.candidate(perturbations)
      return -1 * timeit(lambda: self.model.translate(candidate), number=1)
    return _objective

####Invsible Character Attack

In [ ]:
class InvisibleCharacterSpongeObjective(SpongeObjective, InvisibleCharacterObjective):
  pass

#### Homoglyph Attack

In [ ]:
class HomoglyphSpongeObjective(SpongeObjective, HomoglyphObjective):
  pass

####Reordering Attack

In [ ]:
class ReorderSpongeObjective(SpongeObjective, ReorderObjective):
  pass

####Deletion Attack

In [ ]:
class DeletionSpongeObjective(SpongeObjective, DeletionObjective):
  pass

#### Attack Performance

#####Invisible Character Experiment

In [ ]:
experiment(en2fr, InvisibleCharacterSpongeObjective, source_small, perturbs, 1, 6, 'invisibles_sponge.pkl', 3, 16)

##### Homoglyph Experiment

In [ ]:
experiment(en2fr, HomoglyphSpongeObjective, source_small, perturbs, 1, 6, 'homoglyph_sponge.pkl', 3, 16)

#####Reordering Experiment

In [ ]:
experiment(en2fr, ReorderSpongeObjective, source_small, perturbs, 1, 6, 'reorder_sponge.pkl', 3, 16)

####Deletion Experiment

In [ ]:
experiment(en2fr, DeletionSpongeObjective, source_small, perturbs, 1, 6, 'deletion_sponge.pkl', 3, 16)